In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
from langchain_community.utilities import SQLDatabase
#Chinook is a sample database provided by Microsoft that contains information about a digital media store, including customers, invoices, and tracks. 
# It is often used for testing and learning purposes in database management and SQL querying.
# This code snippet demonstrates how to connect to the Chinook database using the SQLDatabase utility from the langchain_community package. 
# The connection is established using a SQLite URI, which points to the location of the Chinook database file. Once connected, you can perform various database operations such as querying and retrieving data from the Chinook database.

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

In [10]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[sql_query]
)

In [12]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [13]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='ce175989-bd8f-4a3a-93bf-d6864e04150a'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 410, 'prompt_tokens': 142, 'total_tokens': 552, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EW2JHXm6ImdmBi6ouJ32cNSmlbqek', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1120b-2774-7ab3-84e1-0ca3f83c8cec-0', tool_calls=[{'name': 'sql_query', 'args': {'query': 'SHOW TABLES;'}, 'id': 'call_paFEbOZxVfxbQAZNTZNeU5tn', 'type': 'tool_call'}], invalid_tool_ca

In [14]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT a.Name, COUNT(t.TrackId) AS TrackCount
FROM Artist a
JOIN Album al ON al.ArtistId = a.ArtistId
JOIN Track t ON t.AlbumId = al.AlbumId
WHERE a.Name LIKE 'S%' OR a.Name LIKE 's%'
GROUP BY a.Name
ORDER BY TrackCount DESC
LIMIT 1;
